**`US_curate_footprints`**

This pipeline builds a footprint-level building inventory for hurricane damage modeling in the United States.

It runs the complete recipe chain: ingest, harmonize, enrich, and curate.

Parcels are curated first (`US_parcel-openplaces-2026`) to get a land-use classification for use in footprint curation.

Recipe: `US_footprint-cheer-2026.yaml`

File: `src/openplaces/recipes/US/_all/footprint/cheer/2026/US_footprint-cheer-2026.yaml`

Pass `--include_streetview` to include Google Street View (ingest and `n_stories`
enrichment); billed per request, off by default.

Pass `--include_googlesatellite` to include Google Satellite (ingest and roof-shape
enrichment); billed per request, off by default.

Pass `--include_placeslab` to include the placeslab-fmv2026 legacy parcel enrichment lane (ingest of a Google Drive dataset, plus its area-weighted crosswalk enrichment onto current parcels); off by default.



# Configure

In [ ]:
import argparse

from openplaces.core.schema import AdminId
from openplaces.diagnostics import find_recipes
from openplaces.geo.link import create_entity_link
from openplaces.io.cleanup import cleanup
from openplaces.io.curator import curate
from openplaces.io.enricher import enrich
from openplaces.io.harmonizer import harmonize
from openplaces.io.ingester import ingest
from openplaces.recipe import find_entity_recipe_id, get_output_path, get_recipe_by_id
from openplaces.timing import get_timer

In [ ]:
parser = argparse.ArgumentParser(
    description='Ingest, harmonize, enrich, and curate footprints'
)
parser.add_argument(
    '--recipe_id',
    help='Curation recipe (e.g. "US_footprint-cheer-2026")',
)
parser.add_argument(
    '--admin_ids',
    help='Admin unit IDs to process (e.g. "US-NC-BR")',
    nargs='*',
)
parser.add_argument(
    '--reprocess',
    help=(
        "Reprocess admin IDs even if output exists. Bare '--reprocess' "
        "(= 'all') reruns every stage; '--reprocess attributes' reruns "
        'only the attribute/enrich/curate recipes, leaving the geometry '
        'recipes (the geospines) to the output-exists skip and never '
        'forcing a re-ingest (which would touch the very mtimes the '
        'recipe DAG watches)'
    ),
    nargs='?',
    const='all',
    choices=['all', 'attributes'],
    default=None,
)
parser.add_argument(
    '--redownload',
    help='Redownload input data from original source',
    action='store_true',
)
parser.add_argument(
    '--keep_unzipped',
    help='Keep unzipped datasets in heap folder after processing',
    action='store_true',
)
parser.add_argument(
    '--include_streetview',
    help=(
        'Include Google Street View (ingest and the n_stories enrichment '
        'step that depends on it); billed per request, off by default'
    ),
    action='store_true',
)
parser.add_argument(
    '--include_googlesatellite',
    help=(
        'Include Google Satellite (ingest and the roof-shape enrichment '
        'step that depends on it); billed per request, off by default'
    ),
    action='store_true',
)
parser.add_argument(
    '--include_placeslab',
    help=(
        'Include the placeslab-fmv2026 legacy parcel enrichment lane '
        '(ingest of a large, slow, opt-in-only Google Drive dataset, plus '
        'its area-weighted crosswalk enrichment onto current parcels); '
        'off by default'
    ),
    action='store_true',
)
parser.add_argument(
    '--cleanup',
    help=(
        'Reclaim consumed intermediate files as stages finish: "consumed" '
        'deletes cache parquets (and image caches when opted in via '
        'retention.cleanup.include_images) once all their consumers are '
        'complete; "aggressive" additionally treats core spines as '
        'reclaimable once the curated outputs exist'
    ),
    default='none',
    choices=['none', 'consumed', 'aggressive'],
)
parser.add_argument(
    '--verbose',
    action='store_true',
)

# Test arguments

In [ ]:
# Quick state-level county map (to pick `admin_ids`)

from openplaces import get_admin

admin3 = get_admin('US-NC', 3, geom=True)
admin3.explore()

List of arguments passed to the Python script version of this notebook.
Don't forget the spaces after each part of the argument.

In [ ]:
ARGS_TEST = (
    '--recipe_id US_footprint-cheer-2026 '
    '--admin_ids US-NC-CW '  # Chowan county (pilot)
    # '--admin_ids US-PA-DA '  # Dauphin county
    # '--admin_ids US-NC-AR '  # Carteret county (pilot)
    # '--admin_ids US-NC-AR-MO '  # Morehead township (smaller set for testing)
    # '--admin_ids US-NC-BR '  # Brunswick county
    # '--admin_ids US-NC-BR-SH '  # Shallotte township (smaller set for testing)
    # '--admin_ids US-NC-BR US-NC-CI US-NC-DR US-NC-HD US-NC-NH US-NC-ON US-NC-PE '
    # Bristol, Essex, Middlesex, Norfolk, Plymouth, Suffolk (Boston), Worcester
    # '--admin_ids US-MA-BR US-MA-ESS US-MA-MI US-MA-NOK US-MA-PLY US-MA-SU US-MA-WOR '
    # '--admin_ids US-MA-SOM '  # Somerville
    # '--admin_ids US-MA-MI '  # Middlesex county
    # '--admin_ids US-MA-NEW '  # Newton
    # '--admin_ids US-MA-CAM '  # Cambridge
    # '--admin_ids US-MA-SU '  # Suffolk county (Boston)
    # '--admin_ids US-MA-MI '
    # '--admin_ids US-MA-MI US-MA-BR US-MA-ESS US-MA-NOK US-MA-PLY US-MA-SU US-MA-WOR '
    # '--admin_ids US-FL-AL '  # Alachua
    # '--admin_ids US-FL-LA '  # Lake
    # '--admin_ids US-TX-JEF '
    '--reprocess '
    # '--redownload '
    # '--keep_unzipped '
    # '--include_googlesatellite '
    # '--include_streetview '
    # '--include_placeslab '
    # '--cleanup consumed '
    '--verbose '
)

args_list = [x for x in ARGS_TEST.split(' ') if x]
args = parser.parse_args(args_list)
args

In [ ]:
# Pretty-print recipes
from openplaces.utils import pretty_print

curation_recipe = get_recipe_by_id(args.recipe_id)
harmonization_recipe_id = curation_recipe['entity_recipe']
enrichment_recipe_ids = [
    spec['recipe_id']
    for step in curation_recipe['pipeline']
    for spec in step.get('recipes', [])
]
# Parcel curation lane consumed by the footprint recipe's link_curated_entity
# step. Parcels are curated first; the footprint recipe joins their land-use
# (e.g. the manufactured_home_park flag) by parcel_id_local.
parcel_curation_recipe_id = next(
    (
        step['recipe_id']
        for step in curation_recipe['pipeline']
        if step['step'] == 'link_curated_entity'
    ),
    None,
)
parcel_harmonization_recipe_id = (
    get_recipe_by_id(parcel_curation_recipe_id)['entity_recipe']
    if parcel_curation_recipe_id
    else None
)

print('Footprint curation recipe (produces final output):\n')
pretty_print(curation_recipe)

print('\nPrecursor recipes:\n\nFootprint harmonization recipe:\n')
pretty_print(get_recipe_by_id(harmonization_recipe_id))
if parcel_harmonization_recipe_id:
    print('\nParcel harmonization recipe:')
    pretty_print(get_recipe_by_id(parcel_harmonization_recipe_id))
for enrichment_recipe_id in enrichment_recipe_ids:
    print('\nEnrichment recipe:')
    pretty_print(get_recipe_by_id(enrichment_recipe_id))
if parcel_curation_recipe_id:
    print('\nParcel curation recipe:')
    pretty_print(get_recipe_by_id(parcel_curation_recipe_id))

# Ingest precursor datasets

In [ ]:
curation_recipe = get_recipe_by_id(args.recipe_id)
harmonization_recipe_id = curation_recipe['entity_recipe']
enrichment_recipe_ids = [
    spec['recipe_id']
    for step in curation_recipe['pipeline']
    for spec in step.get('recipes', [])
]
# Parcel curation lane (curated before footprints; see link_curated_entity).
parcel_curation_recipe_id = next(
    (
        step['recipe_id']
        for step in curation_recipe['pipeline']
        if step['step'] == 'link_curated_entity'
    ),
    None,
)
parcel_harmonization_recipe_id = (
    get_recipe_by_id(parcel_curation_recipe_id)['entity_recipe']
    if parcel_curation_recipe_id
    else None
)

# --reprocess semantics: 'all' is the full rebuild; 'attributes'
# reruns only the attribute-phase harmonize recipes plus enrich/curate,
# leaving the geometry recipes (geospines) to the output-exists skip.
reprocess_geometry = args.reprocess == 'all'
reprocess_attributes = args.reprocess in ('all', 'attributes')


# A harmonize recipe may be split into a geometry recipe (geospine) and
# an attribute recipe that declares it via entity_recipe. Walk that chain
# so the driver runs the geometry phase first; an unsplit recipe yields a
# one-element chain and behaves exactly as before.
def harmonize_chain(recipe_id):
    chain = []
    current = recipe_id
    while current:
        recipe = get_recipe_by_id(current)
        if recipe.get('stage') != 'harmonize':
            break
        chain.append(current)
        current = recipe.get('entity_recipe')
    return list(reversed(chain))


def harmonize_reprocess(recipe_id, attribute_recipe_id):
    # The chain tail (the named recipe) is the attribute phase; every
    # predecessor is geometry. An unsplit recipe is its own tail, so
    # 'attributes' reruns it whole -- it cannot split the work.
    if recipe_id == attribute_recipe_id:
        return reprocess_attributes
    return reprocess_geometry


# Save keyword arguments that will be passed to all ingest functions
ingest_kwargs = {
    'reprocess': reprocess_geometry,
    'redownload': args.redownload,
    'keep_unzipped': args.keep_unzipped,
    'verbose': args.verbose,
}

# Harmonize and curate run at the recipes' process level (admin level 3).
# Truncate finer-grained admin IDs (e.g. a township) to their county;
# ingest and enrich calls below take args.admin_ids directly: ingest
# self-truncates (keeping the requested level for image recipes), and
# enrich restricts image-based steps to the requested units.
process_admin_ids = list(
    dict.fromkeys(str(AdminId(*AdminId(a).levels[:3])) for a in args.admin_ids)
)

# Track stage runtimes; saved to the logs directory at the end of the run
timer = get_timer(
    'US_curate_footprints',
    admin_id=process_admin_ids[0] if len(process_admin_ids) == 1 else None,
    verbose=args.verbose,
    overwrite=True,
    recipe_id=args.recipe_id,
    admin_ids=args.admin_ids,
)

## Admin boundaries
One-time downloads (ignoring ``reprocess`` and ``redownload`` args.)

In [ ]:
# US admin boundaries (for allocating Microsoft footprints to counties)
ingest('US_admin-census-2021_admin2', verbose=args.verbose)
timer.mark('ingest US_admin-census-2021_admin2')

ingest('US_admin-census-2021_admin3', verbose=args.verbose)
timer.mark('ingest US_admin-census-2021_admin3')

# Image recipes fetch at admin level 4 (townships)
ingest('US_admin-census-2021_admin4', verbose=args.verbose)
timer.mark('ingest US_admin-census-2021_admin4')

In [ ]:
# Create the harmonized openplaces admin dataset (required to
# ingest Overture data)
harmonize('admin-openplaces-2026_admin3', verbose=args.verbose)
timer.mark('harmonize admin boundaries')

## Tiles

In [ ]:
# Tile-partitioned recipes (e.g. footprint-obm-2025) resolve which tiles
# to download for an admin unit via a precomputed tile <-> admin overlay
# link
obm_recipe = get_recipe_by_id('footprint-obm-2025')
tile_recipe_id = obm_recipe['download_by']['tile_recipe_id']
admin_recipe_id = obm_recipe['overlay_admin_ids']['admin_recipe_id']

ingest(tile_recipe_id, verbose=args.verbose)
timer.mark(f'ingest {tile_recipe_id}')

tiles_path = get_output_path(get_recipe_by_id(tile_recipe_id))
tile_admin_link_path = tiles_path.with_name(
    tiles_path.stem + f'_{admin_recipe_id}.parquet'
)
if not tile_admin_link_path.exists():
    create_entity_link(tile_recipe_id, admin_recipe_id)
    timer.mark(f'link {tile_recipe_id} to {admin_recipe_id}')

## Footprints

In [ ]:
# Global building footprints: OpenBuildingsMap (OBM)
ingest('footprint-obm-2025', admin_ids=args.admin_ids, **ingest_kwargs)
timer.mark('ingest footprint-obm-2025')

In [ ]:
# US building footprints: Microsoft
ingest('US_footprint-microsoft-v2', admin_ids=args.admin_ids, **ingest_kwargs)
timer.mark('ingest US_footprint-microsoft-v2')

In [ ]:
# State footprints: auto-discovered per state from args.admin_ids
state_groups = {}
for aid_str in args.admin_ids or []:
    aid = AdminId(aid_str)
    if aid.get_level() >= 2:
        state_id = str(AdminId(*aid.levels[:2]))
        state_groups.setdefault(state_id, []).append(aid_str)

for state_id, children in state_groups.items():
    recipe_id = find_entity_recipe_id(
        state_id, 'footprint', stage='ingest', silent=True
    )
    if recipe_id:
        ingest(recipe_id, admin_ids=children, **ingest_kwargs)
timer.mark('ingest state footprints')

In [ ]:
# US building footprints: FEMA USA Structures.
# Ingested for parcel-occupancy linkage, not as a footprint-spine geometry source
# (FEMA caused IoU-merge errors). The parcel spine attributes FEMA's dominant
# occupancy per parcel; that flows to footprints as occupancy evidence.
ingest('US_footprint-fema-2023', admin_ids=args.admin_ids, **ingest_kwargs)
timer.mark('ingest US_footprint-fema-2023')

## Parcels

In [ ]:
# Parcels: enumerate every ingest-stage parcel recipe whose admin scope
# covers a requested admin unit (not just one state-wide winner --
# find_entity_recipe_id truncated to a state_id can only find recipes filed
# at that state level or coarser, and it's a single-winner lookup, so it
# can't be reused here: a county can have more than one parcel-entity ingest
# recipe, e.g. Dauphin County's geometry recipe (parcel-dauphinco-2026) plus
# its separate, county-scoped attribute recipe (parcel-dauphinco-2026pc)).
parcel_recipes = find_recipes('parcel', stage='ingest')
recipe_admin_ids = {}
for aid_str in args.admin_ids or []:
    aid = AdminId(aid_str)
    for _, row in parcel_recipes.iterrows():
        if row['admin_id'] and AdminId(row['admin_id']).is_parent_or_equal_of(aid):
            recipe_id = f'{row["admin_id"]}_parcel-{row["source_id"]}-{row["version"]}'
            recipe_admin_ids.setdefault(recipe_id, []).append(aid_str)

for recipe_id, children in recipe_admin_ids.items():
    ingest(recipe_id, admin_ids=children, **ingest_kwargs)
timer.mark('ingest parcels')

In [ ]:
# Legacy "placeslab-fmv2026" parcel dataset: opt-in reference data (a large,
# slow Google Drive download) used only for the area-weighted crosswalk
# enrichment below (US_parcel_parcel-placeslab-fmv2026).
if args.include_placeslab:
    ingest('US_parcel-placeslab-fmv2026', admin_ids=args.admin_ids, **ingest_kwargs)
    timer.mark('ingest US_parcel-placeslab-fmv2026')

## Properties

In [ ]:
# Property/tax-roll recipes (e.g. US-PA-DA_property-dauphinco-2026,
# US-FL_property-fldor-2026): county-scoped attribute-only sources with no
# footprint/parcel geometry of their own, so unlike footprints/parcels above
# they're never reachable via find_entity_recipe_id's admin-hierarchy walk
# from a state_id. Ingested here so harmonize's own auto-discovery
# (_discover_link_sources) has something on disk to find and join onto the
# parcel spine as evidence.
property_recipes = find_recipes('property', stage='ingest')
recipe_admin_ids = {}
for aid_str in args.admin_ids or []:
    aid = AdminId(aid_str)
    for _, row in property_recipes.iterrows():
        if row['admin_id'] and AdminId(row['admin_id']).is_parent_or_equal_of(aid):
            recipe_id = (
                f'{row["admin_id"]}_property-{row["source_id"]}-{row["version"]}'
            )
            recipe_admin_ids.setdefault(recipe_id, []).append(aid_str)

for recipe_id, children in recipe_admin_ids.items():
    ingest(recipe_id, admin_ids=children, **ingest_kwargs)
timer.mark('ingest properties')

## Buildings

In [ ]:
# US building points: National Structure Inventory (NSI)
ingest('US_building-nsi-2026', admin_ids=args.admin_ids, **ingest_kwargs)
timer.mark('ingest US_building-nsi-2026')

## Dwellings (address points)

In [ ]:
# Global dwelling points: Overture
ingest('dwelling-overture-2025', admin_ids=args.admin_ids, **ingest_kwargs)
timer.mark('ingest dwelling-overture-2025')

# Harmonize

Harmonize the footprint spine, then the parcel spine (which reads the footprint
spine to summarize per-parcel footprint morphology).

Each spine is a two-recipe chain: the geospine (geometry phase, persisted link
tables) runs first, then the attribute recipe restores its results and attaches
evidence with no spatial computation. `--reprocess attributes` reruns only the
attribute half.

## Harmonize footprints

In [ ]:
for _recipe_id in harmonize_chain(harmonization_recipe_id):
    harmonize(
        _recipe_id,
        admin_ids=process_admin_ids,
        reprocess=harmonize_reprocess(_recipe_id, harmonization_recipe_id),
        verbose=args.verbose,
    )
timer.mark('harmonize')

## Harmonize parcels

In [ ]:
# Harmonize the parcel spine: links parcels to local tax records and the NSI
# building group, and attaches per-parcel footprint morphology (reads the
# footprint spine above). This is the input to the parcel curation lane.
if parcel_harmonization_recipe_id:
    for _recipe_id in harmonize_chain(parcel_harmonization_recipe_id):
        harmonize(
            _recipe_id,
            admin_ids=process_admin_ids,
            reprocess=harmonize_reprocess(_recipe_id, parcel_harmonization_recipe_id),
            verbose=args.verbose,
        )
    timer.mark('harmonize parcels')

# Both spines exist now, so the ingested cache parquets they consumed
# (footprints, parcels, NSI, Overture) have no incomplete consumers left;
# reclaim them (each deletion leaves a tombstone receipt so a rerun still
# skips the ingest).
if args.cleanup != 'none':
    cleanup(
        args.recipe_id,
        admin_ids=process_admin_ids,
        stages=('ingest',),
        dry_run=False,
        verbose=args.verbose,
    )
    timer.mark('cleanup ingested inputs')

# Enrich

## Ingest images

In [ ]:
# Imagery has no ingest stage. Google's Static API policy prohibits storing
# or caching content, so the enrichment steps below fetch what they need in
# memory and discard it. Nothing is downloaded here.
#
# The two --include_* flags therefore gate live, billed API calls rather than
# a one-off download: with them unset, the image-based enrichment recipes are
# dropped entirely in the next cell.
image_recipe_ids = [
    get_recipe_by_id(recipe_id).get('image_recipe')
    for recipe_id in enrichment_recipe_ids
]
print('Image recipes used by this curation lane:')
for image_recipe_id in dict.fromkeys(image_recipe_ids):
    if image_recipe_id:
        print(f'  {image_recipe_id} (fetched on the fly during enrich)')

## Enrich footprints with images

In [ ]:
if not args.include_streetview:
    # Drop enrichment recipes that depend on Street View entirely. Each
    # enrich run fetches imagery live and is billed per request, so this is
    # what keeps an unwanted image lane from costing money.
    enrichment_recipe_ids = [
        recipe_id
        for recipe_id in enrichment_recipe_ids
        if 'streetview' not in (get_recipe_by_id(recipe_id).get('image_recipe') or '')
    ]

if not args.include_googlesatellite:
    # Drop enrichment recipes that depend on Google Satellite entirely, for
    # the same reason: imagery is fetched live and billed per request.
    enrichment_recipe_ids = [
        recipe_id
        for recipe_id in enrichment_recipe_ids
        if 'googlesatellite'
        not in (get_recipe_by_id(recipe_id).get('image_recipe') or '')
    ]

for enrichment_recipe_id in enrichment_recipe_ids:
    enrich(
        enrichment_recipe_id,
        admin_ids=args.admin_ids,
        entity_recipe_id=harmonization_recipe_id,
        reprocess=reprocess_attributes,
        verbose=args.verbose,
    )
    timer.mark(f'enrich {enrichment_recipe_id}')

# All enrichment evidence is saved; image caches are reclaimable once every
# enrich recipe sharing an image_recipe has full coverage. Deletion of the
# caches themselves stays opt-in (retention.cleanup.include_images) — the
# report always lists them.
if args.cleanup != 'none':
    cleanup(
        args.recipe_id,
        admin_ids=process_admin_ids,
        dry_run=False,
        verbose=args.verbose,
    )
    timer.mark('cleanup enrichment inputs')

In [ ]:
# Legacy "placeslab-fmv2026" parcel dataset: area-weighted crosswalk evidence
# (elevation, land value, etc.) onto current parcels. Opt-in; see the
# matching ingest cell under "## Parcels" above.
if args.include_placeslab and parcel_harmonization_recipe_id:
    enrich(
        'US_parcel_parcel-placeslab-fmv2026',
        admin_ids=args.admin_ids,
        entity_recipe_id=parcel_harmonization_recipe_id,
        reprocess=reprocess_attributes,
        verbose=args.verbose,
    )
    timer.mark('enrich US_parcel_parcel-placeslab-fmv2026')

# Curate

Curate parcels first (land-use classification), then footprints — the footprint
recipe consumes the curated parcel use group via `link_curated_entity`.

## Curate parcels

In [ ]:
# Curate parcels first: classify land use (manufactured-home park vs RV park vs
# other) from assessor codes, the linked-NSI group, and footprint morphology.
# The footprint curate below joins this in via link_curated_entity.
if parcel_curation_recipe_id:
    curate(
        parcel_curation_recipe_id,
        admin_ids=process_admin_ids,
        reprocess=reprocess_attributes,
        verbose=args.verbose,
    )
    timer.mark('curate parcels')

## Curate footprints

In [ ]:
curate(
    args.recipe_id,
    admin_ids=process_admin_ids,
    reprocess=reprocess_attributes,
    verbose=args.verbose,
    save_statistics=True,
)
timer.mark('curate')

# Final sweep now that the curated deliverable exists. 'aggressive'
# additionally treats the core spines as reclaimable (kept only until the
# curated outputs exist); enrichment evidence sidecars are always kept —
# their image inputs may be gone.
if args.cleanup != 'none':
    cleanup(
        args.recipe_id,
        admin_ids=process_admin_ids,
        aggressive=(args.cleanup == 'aggressive'),
        dry_run=False,
        verbose=args.verbose,
    )
    timer.mark('cleanup consumed intermediates')

In [ ]:
# Report and save stage runtimes (JSON in the logs directory)
timer.summary()
timer.save()

---
# Convert to script

*The above line and heading identify the end of the script.*

*Code below this marker will not be included in the converted `.py` script.*

In [ ]:
from openplaces.flow import convert_to_script

COMMIT = True
# If True, writes `.py` scripts to 'scripts/'.
# If False, writes a test version of the script to 'scripts/_test/'

In [ ]:
convert_to_script(commit=COMMIT)

# Test script

In [ ]:
# test_script(*args_list, committed=COMMIT)

# Loop script

In [ ]:
# CHEER_ADMIN3_IDS = 'US-NC-AR US-TX-JEF US-FL-LA US-MA-SOM'.split()
# CHEER_ADMIN3_IDS = 'US-NC-BE US-NC-BT US-NC-BL US-NC-BR US-NC-CM US-NC-AR US-NC-CW US-NC-CO US-NC-CR US-NC-CU US-NC-CI US-NC-DR US-NC-DP US-NC-ED US-NC-FR US-NC-GT US-NC-GE US-NC-HL US-NC-HA US-NC-HR US-NC-HO US-NC-HD US-NC-JO US-NC-JN US-NC-LE US-NC-LN US-NC-MR US-NC-NA US-NC-NH US-NC-NO US-NC-ON US-NC-PM US-NC-PA US-NC-PE US-NC-PQ US-NC-PI US-NC-RB US-NC-SA US-NC-SC US-NC-TY US-NC-WA US-NC-WR US-NC-WS US-NC-WY US-NC-WI'.split()
# CHEER_ADMIN3_IDS = 'US-TX-ARA US-TX-AUS US-TX-BEE US-TX-BRA US-TX-BRK US-TX-CAN US-TX-CAM US-TX-CHA US-TX-CLR US-TX-DEW US-TX-DUV US-TX-FAY US-TX-FOB US-TX-GAL US-TX-GOL US-TX-HRD US-TX-HAR US-TX-HID US-TX-JAC US-TX-JAS US-TX-JEF US-TX-JIH US-TX-JIW US-TX-KEY US-TX-KLE US-TX-LAV US-TX-LIB US-TX-LIO US-TX-MAT US-TX-NEW US-TX-NUE US-TX-ORA US-TX-REF US-TX-SAP US-TX-STA US-TX-TYL US-TX-VIC US-TX-WAR US-TX-WAS US-TX-WEB US-TX-WHA US-TX-WIY'.split()

In [ ]:
# args_list_cheer = (
#     ['--recipe_id', args.recipe_id, '--admin_ids']
#     + CHEER_ADMIN3_IDS
#     + ['--reprocess', '--verbose']
# )
# args_list_cheer

In [ ]:
# from openplaces.flow import test_script

# test_script(*args_list_cheer)

# Export the region-wide delivery bundle

Pools the per-county curated files into one shareable set for the whole region.

The set is four files, not one. They share the `footprint_id` index in the same
row order, so any two of them rejoin 1:1 and a consumer loads only the part
they need:

- `US-NC_footprint-cheer-2026.parquet` -- the canonical attributes, as a plain
  table. This is the file to hand to a modeller.
- `US-NC_footprint-cheer-2026_point.parquet` -- the same attributes on centroid
  points, plus `structure_value_per_area`. Small enough to map directly.
- `US-NC_footprint-cheer-2026_geo.parquet` -- the footprint polygons, nothing
  else. Only needed when the outline itself matters.
- `US-NC_footprint-cheer-2026_evidence.parquet` -- every remaining curated
  column: the source-attributed evidence the canonical values were reconciled
  from.

Nothing is configured here. The recipe's `share:` block names both the
canonical columns and the region's 45 counties, so the same call works from
this notebook and from the Snakemake pipeline, which runs it as its terminal
`deliver` job.

The four files are left read-only, since they are what leaves the repository.
Re-running this cell unlocks and rewrites them.

QGIS opens the set through the `Load joined openplaces parquet files`
algorithm: pick any file of the four, choose polygons or centroid points, and
tick "Join evidence columns" when the supplement is wanted too.

In [ ]:
from openplaces.api import export_delivery

# Reads each county twice (canonical + geometry, then evidence) so the wide
# evidence columns are never in memory alongside 2.6M polygons.
paths = export_delivery(args.recipe_id, verbose=True)
paths

## Verify the bundle

Confirms the four files line up before anything is copied out of the repo.

Checks that every file carries the same `footprint_id` index, that the
canonical attributes are populated, and that the centroids fall inside the
region.

In [ ]:
import pandas as pd

from openplaces.io import read_parquet

canonical = read_parquet(paths['canonical'])
evidence = read_parquet(paths['evidence'])
point = read_parquet(paths['point'], geom=True)
geo = read_parquet(paths['geo'], geom=True)

for name, frame in [('point', point), ('geo', geo), ('evidence', evidence)]:
    assert frame.index.equals(canonical.index), f'{name} index does not match'
    assert not frame.index.has_duplicates, f'{name} has duplicate ids'

print(f'{len(canonical):,} footprints, indexes aligned across all four files')
print(f'{point.total_bounds.round(3)} point bounds')
print(f'{geo.crs} polygon CRS')

pd.options.display.max_rows = canonical.shape[1] + 5
canonical.notna().mean().to_frame('fill_rate')

# Inspect outputs

In [ ]:
import pandas as pd

from openplaces import get_entities

## Footprints

In [ ]:
footprints = get_entities(args.recipe_id, args.admin_ids, geom=True)
pd.options.display.max_rows = max(len(footprints.columns), 50)
print(len(footprints))
footprints.sample(5).T

### Show a random footprint

In [ ]:
from openplaces.io.curator import Curator

curator = Curator(args.recipe_id, admin_ids=process_admin_ids, verbose=args.verbose)
curator.show_random_entity()

### Inspect building imagery

For a randomly sampled building, show its footprint in context alongside the
downloaded Google Satellite and Street View images that fed the enrichment
detectors. Panels show a "not available" placeholder when no image was
downloaded for that building.

In [ ]:
from openplaces.viz import show_building_imagery

if args.include_googlesatellite or args.include_streetview:
    # Image recipes used by this curation recipe's enrichment steps, with a
    # friendly panel label per image source (satellite + street view).
    LABELS = {'googlesatellite': 'Google Satellite', 'googlestreetview': 'Street View'}
    image_recipes = {}
    for enrichment_recipe_id in enrichment_recipe_ids:
        image_recipe_id = get_recipe_by_id(enrichment_recipe_id).get('image_recipe')
        if not image_recipe_id:
            continue
        label = next(
            (name for key, name in LABELS.items() if key in image_recipe_id),
            image_recipe_id,
        )
        image_recipes[label] = image_recipe_id

    # Context map + each building's downloaded Google imagery (placeholder when an
    # image was not downloaded, e.g. Street View if its ingest was skipped).
    footprints = get_entities(args.recipe_id, args.admin_ids, geom=True)
    building = footprints.sample(1)

    show_building_imagery(
        location=building,
        geodatasets={'footprints': footprints},
        image_recipes=image_recipes,
        admin_id=args.admin_ids[0],
    )

## Parcels

In [ ]:
parcels = get_entities(parcel_curation_recipe_id, args.admin_ids[0], geom=True)
pd.options.display.max_rows = max(len(parcels.columns), 50)
print(len(parcels))
parcels.sample(5).T

In [ ]:
import numpy as np

parcels['land_value_per_area_log'] = (
    parcels['land_value'].div(parcels['area_ha']).apply(np.log)
)
ax = parcels.plot(
    'land_value_per_area_log',
    # scheme='quantiles',
    cmap='RdYlGn_r',
    # k=25,
    vmin=8.5,
    vmax=15,
    figsize=(12, 12),
    legend=True,
)
ax.axis('off')

# Profile imagery disk usage

The image caches written by this notebook can grow large (tens of GB per
county).

In [ ]:
# from openplaces import diagnostics

# # Disk usage by admin unit and dataset across the configured data
# # directories (core, external, heap, cache, out)
# usage = diagnostics.profile_disk_usage(min_size_mb=50)
# usage.head(15)

In [ ]:
# Image caches by location
# diagnostics.list_image_caches()

In [ ]:
# Delete with a county or township ID (dry_run=True only reports;
# pass dry_run=False to actually delete)
# from openplaces.io import delete_image_caches
# delete_image_caches(['US-NC-BR'], dry_run=True)
# delete_image_caches(args.admin_ids, dry_run=True)

# Show a random footprint

In [ ]:
from openplaces.io.curator import Curator

curator = Curator(args.recipe_id, admin_ids=process_admin_ids, verbose=args.verbose)
curator.show_random_entity()

# Export QGIS maps

In [ ]:
# # One .qgz per curated county (export_qgis_map resolves a single admin_id's
# # worth of layers; process_admin_ids is what curate() above actually wrote).
# from openplaces import export_qgis_map

# for admin_id in process_admin_ids:
#     qgz_path = export_qgis_map(args.recipe_id, admin_id, verbose=args.verbose)
#     print(f'Wrote {qgz_path}')